<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;"><div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div><div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div></div>
<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;"><h1 style="margin:0;"><b>Filtering in Spatial Domain — Algorithm</b></h1></div>


## Purpose and reading order

This notebook is the algorithm specification for the corresponding `main.ipynb` in this lab. It has two linked layers:

| Layer | Content | Question answered |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained executable-style block with explicit branches and gates | What is the whole pipeline? |
| **Part II — Auditable specification** | Six-section contract for every step: Purpose, Inputs, Procedure, Produces, Acceptance, Failure mode | Why is each step valid? |

Names, shapes, thresholds, and failure decisions are kept identical between both layers. The acceptance contract of step *k* is the precondition of step *k+1*.

## Part I — Complete pseudocode

The single block below is the complete pipeline. Every named artifact is assigned before use; every failure is explicit. It is a direct, implementation-neutral transcription of the executable notebook.

```text
ALGORITHM Spatial_Domain_Filtering

# Algorithm 1 — Setup — Environment and Configuration
PLOT.rcParams["figure.dpi"] ← 110
PLOT.rcParams["axes.titlesize"] ← 11

# Algorithm 2 — 1. Data and Output Paths
PROCEDURE locate_lab_root(start)
    start ← start.resolve()
    FOR candidate IN [start, EXPAND start.parents] DO
        IF ((candidate / "data").is_dir() AND (candidate / "notebooks").is_dir()) THEN
            RETURN candidate
        END IF
    END FOR
    RAISE FileNotFoundError("Could not locate the lab root containing data/ and notebooks/.")
END PROCEDURE
LAB_DIR ← locate_lab_root(PATH.cwd())
DATA_DIR ← (LAB_DIR / "data")
OUTPUT_DIR ← ((LAB_DIR / "outputs") / "figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
IMAGE_EXTENSIONS ← {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES ← SORT(STREAM[path FOR path IN DATA_DIR.rglob("*") WHERE (path.is_file() AND (path.suffix.lower() IN IMAGE_EXTENSIONS))])
REQUIRE DATA_FILES OTHERWISE FORMAT("No supported images found in: {0}" , DATA_DIR)
PROCEDURE select_input_image(VARIADIC tokens)
    tokens ← TUPLE(STREAM[token.lower() FOR token IN tokens])
    matches ← LIST[path FOR path IN DATA_FILES WHERE ALL(STREAM[(token IN path.relative_to(DATA_DIR).as_posix().lower()) FOR token IN tokens])]
    IF (NOT matches) THEN
        RAISE FileNotFoundError(FORMAT("No image matching {0} found in {1}" , tokens, DATA_DIR))
    END IF
    RETURN matches[0]
END PROCEDURE
ROLE_PATTERNS ← {"moon" ↦ ("moon-blurred",), "ascent" ↦ ("ascentb",), "einstein_ref" ↦ ("einstein/ref",),
    "einstein_gaussian" ↦ ("einstein_gaussian",), "einstein_salt" ↦ ("saltandpepper",),
    "einstein_speckle" ↦ ("speckle",), "taj_ref" ↦ ("tajmahal/ref",), "zebra_ref" ↦ ("zebra/ref",),
    "monarch_ref" ↦ ("monarch/ref",)}
INPUT_IMAGES ← MAP{role ↦ select_input_image(EXPAND patterns) FOR (role, patterns) IN ITEMS(ROLE_PATTERNS)}
CALL EMIT("Lab directory :", LAB_DIR)
CALL EMIT("Data directory:", DATA_DIR)
CALL EMIT("Images discovered:", LENGTH(DATA_FILES))
CALL EMIT("Experiment roles :", LENGTH(INPUT_IMAGES))
CALL EMIT("Output folder :", OUTPUT_DIR)

# Algorithm 3 — Data Inventory
PROCEDURE load_grayscale_image(path)
    RETURN ARRAY.array(IMAGE.open(path).convert("L"), dtype = ARRAY.uint8)
END PROCEDURE
PROCEDURE load_rgb_image(path)
    RETURN ARRAY.array(IMAGE.open(path).convert("RGB"), dtype = ARRAY.uint8)
END PROCEDURE
einstein_reference_image ← load_grayscale_image(INPUT_IMAGES["einstein_ref"])
einstein_gaussian_noise_image ← load_grayscale_image(INPUT_IMAGES["einstein_gaussian"])
einstein_salt_pepper_noise_image ← load_grayscale_image(INPUT_IMAGES["einstein_salt"])
einstein_speckle_noise_image ← load_grayscale_image(INPUT_IMAGES["einstein_speckle"])
moon ← load_grayscale_image(INPUT_IMAGES["moon"])
ascent ← load_grayscale_image(INPUT_IMAGES["ascent"])
taj_reference_image ← load_rgb_image(INPUT_IMAGES["taj_ref"])
zebra_reference_image ← load_rgb_image(INPUT_IMAGES["zebra_ref"])
monarch_reference_image ← load_rgb_image(INPUT_IMAGES["monarch_ref"])
FOR (name, image) IN ITEMS({"einstein_ref" ↦ einstein_reference_image,
    "einstein_gaussian" ↦ einstein_gaussian_noise_image, "einstein_salt" ↦ einstein_salt_pepper_noise_image,
    "einstein_speckle" ↦ einstein_speckle_noise_image, "moon" ↦ moon, "ascent" ↦ ascent,
    "taj_ref" ↦ taj_reference_image, "zebra_ref" ↦ zebra_reference_image,
    "monarch_ref" ↦ monarch_reference_image}) DO
    CALL EMIT(FORMAT("{0:20s} shape={1!s:16s} dtype={2} range=({3}, {4})" , name, image.shape, image.dtype,
        MIN(image), MAX(image)))
END FOR

# Algorithm 4 — 2. Spatial Filtering Formulation
toy ← ARRAY.array([[10, 10, 10, 10, 10], [10, 20, 20, 20, 10], [10, 20, 90, 20, 10], [10, 20, 20, 20, 10],
    [10, 10, 10, 10, 10]], dtype = ARRAY.float32)
center_neighborhood ← toy[(1:4, 1:4)]
CALL EMIT("Toy image:")
CALL EMIT(toy)
CALL EMIT("\n3x3 neighborhood around the center:")
CALL EMIT(center_neighborhood)

# Algorithm 5 — 3. Kernel Properties and Normalization
mean_3x3 ← (ARRAY.ones((3, 3), dtype = ARRAY.float32) / 9.0)
sobel_x_kernel ← ARRAY.array([[(−1), 0, 1], [(−2), 0, 2], [(−1), 0, 1]], dtype = ARRAY.float32)
CALL EMIT("Mean kernel:")
CALL EMIT(mean_3x3)
CALL EMIT("Sum:", SUM(mean_3x3))
CALL EMIT("\nSobel-x kernel:")
CALL EMIT(sobel_x_kernel)
CALL EMIT("Sum:", SUM(sobel_x_kernel))

# Algorithm 6 — 4. Correlation vs Convolution
small ← RESHAPE(ARRAY.arange(1, 26, dtype = ARRAY.float32), 5, 5)
asymmetric_kernel ← ARRAY.array([[1, 2, 0], [0, 0, 0], [0, 0, (−1)]], dtype = ARRAY.float32)
flipped_kernel ← ARRAY.flip(asymmetric_kernel, axis = (0, 1))
CALL EMIT("Input:")
CALL EMIT(small)
CALL EMIT("\nOriginal kernel:")
CALL EMIT(asymmetric_kernel)
CALL EMIT("\nKernel flipped for convolution:")
CALL EMIT(flipped_kernel)
patch ← small[(1:4, 1:4)]
manual_corr ← SUM(ARRAY, (patch * asymmetric_kernel))
manual_conv ← SUM(ARRAY, (patch * flipped_kernel))
CALL EMIT("Patch:")
CALL EMIT(patch)
CALL EMIT("\nManual correlation response:", manual_corr)
CALL EMIT("Manual convolution response :", manual_conv)
scipy_corr ← NDIMAGE.correlate(small, asymmetric_kernel, mode = "constant", cval = 0.0)
scipy_conv ← NDIMAGE.convolve(small, asymmetric_kernel, mode = "constant", cval = 0.0)
CALL EMIT("SciPy correlation at center:", scipy_corr[(2, 2)])
CALL EMIT("SciPy convolution at center :", scipy_conv[(2, 2)])
REQUIRE ARRAY.isclose(manual_corr, scipy_corr[(2, 2)]) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(manual_conv, scipy_conv[(2, 2)]) OTHERWISE STOP WITH ValidationError
CALL EMIT("\nPASS — manual calculations agree with SciPy.")

# Algorithm 7 — 5. Direct Convolution Implementation
PROCEDURE convolve_image_manually(image, kernel)
    image ← ARRAY.asarray(image, dtype = REAL)
    kernel ← ARRAY.asarray(kernel, dtype = REAL)
    (kh, kw) ← kernel.shape
    REQUIRE (((kh MOD 2) = 1) AND ((kw MOD 2) = 1)) OTHERWISE "Kernel dimensions must be odd."
    pad_h ← (kh DIV 2)
    pad_w ← (kw DIV 2)
    padded ← ARRAY.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode = "symmetric")
    kernel_flipped ← ARRAY.flip(kernel, axis = (0, 1))
    output ← ARRAY.zeros_like(image, dtype = REAL)
    FOR row IN RANGE(image.shape[0]) DO
        FOR col IN RANGE(image.shape[1]) DO
            neighbourhood ← padded[(row:(row + kh), col:(col + kw))]
            output[(row, col)] ← SUM(ARRAY, (neighbourhood * kernel_flipped))
        END FOR
    END FOR
    RETURN output
END PROCEDURE
toy_float64 ← ARRAY.asarray(toy, dtype = ARRAY.float64)
kernel_float64 ← ARRAY.asarray(mean_3x3, dtype = ARRAY.float64)
manual_convolution_result ← convolve_image_manually(toy_float64, kernel_float64)
scipy_convolution_result ← NDIMAGE.convolve(toy_float64, kernel_float64, mode = "reflect")
maximum_convolution_difference ← MAX(ARRAY, ARRAY.abs((manual_convolution_result − scipy_convolution_result)))
CALL EMIT("Manual dtype:", manual_convolution_result.dtype)
CALL EMIT("SciPy dtype :", scipy_convolution_result.dtype)
CALL EMIT("Maximum absolute difference:", maximum_convolution_difference)
REQUIRE ARRAY.allclose(manual_convolution_result, scipy_convolution_result, rtol = 1e-10,
    atol = 1e-12) OTHERWISE STOP WITH ValidationError
CALL EMIT("PASS — manual convolution matches scipy.ndimage.convolve.")

# Algorithm 8 — 6. Border Handling
demo ← ARRAY.zeros((9, 9), dtype = ARRAY.float32)
demo[(2:7, 2:7)] ← 200
large_mean_filter_result ← (ARRAY.ones((5, 5), dtype = ARRAY.float32) / 25.0)
border_modes ← {"constant" ↦ NDIMAGE.convolve(demo, large_mean_filter_result, mode = "constant", cval = 0.0),
    "nearest" ↦ NDIMAGE.convolve(demo, large_mean_filter_result, mode = "nearest"),
    "reflect" ↦ NDIMAGE.convolve(demo, large_mean_filter_result, mode = "reflect"),
    "mirror" ↦ NDIMAGE.convolve(demo, large_mean_filter_result, mode = "mirror"),
    "wrap" ↦ NDIMAGE.convolve(demo, large_mean_filter_result, mode = "wrap")}
(fig, axes) ← PLOT.subplots(2, 3, figsize = (11, 7))
axes ← FLATTEN(axes)
CALL axes[0].imshow(demo, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Input")
FOR (ax, (name, result)) IN ZIP(axes[1:], ITEMS(border_modes)) DO
    CALL ax.imshow(result, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(name)
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "01_border_modes.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 9 — 7. Mean / Box Filtering
mean_kernel ← (ARRAY.ones((3, 3), dtype = ARRAY.float32) / 9.0)
mean_filtered_image ← NDIMAGE.convolve(CAST(einstein_gaussian_noise_image, ARRAY.float32), mean_kernel,
    mode = "reflect")
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
CALL axes[0].imshow(einstein_reference_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Reference")
CALL axes[1].imshow(einstein_gaussian_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Gaussian-noisy")
CALL axes[2].imshow(mean_filtered_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("3x3 mean filter")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "02_mean_filter.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 10 — Kernel-Size Sweep
mean_sizes ← [3, 5, 9]
mean_filter_results ← {}
FOR size IN mean_sizes DO
    kernel ← (ARRAY.ones((size, size), dtype = ARRAY.float32) / (size * size))
    mean_filter_results[size] ← NDIMAGE.convolve(CAST(einstein_gaussian_noise_image, ARRAY.float32), kernel,
        mode = "reflect")
END FOR
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
CALL axes[0].imshow(einstein_gaussian_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Noisy input")
FOR (ax, size) IN ZIP(axes[1:], mean_sizes) DO
    CALL ax.imshow(mean_filter_results[size], cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(FORMAT("{0}x{1} mean" , size, size))
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "03_mean_kernel_sizes.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 11 — Gaussian-Kernel Construction
PROCEDURE build_gaussian_kernel_2d(size, sigma)
    REQUIRE ((size MOD 2) = 1) OTHERWISE "Kernel size must be odd."
    REQUIRE (sigma > 0) OTHERWISE "sigma must be positive."
    radius ← (size DIV 2)
    coords ← ARRAY.arange((−radius), (radius + 1), dtype = ARRAY.float64)
    (x, y) ← ARRAY.meshgrid(coords, coords)
    kernel ← ARRAY.exp(((−((x ^ 2) + (y ^ 2))) / (2.0 * (sigma ^ 2))))
    kernel ← (kernel / SUM(kernel))
    RETURN kernel
END PROCEDURE
g5 ← build_gaussian_kernel_2d(5, 1.0)
CALL EMIT(ARRAY.round(g5, 4))
CALL EMIT("Kernel sum:", SUM(g5))
PROCEDURE build_gaussian_kernel_1d(size, sigma)
    radius ← (size DIV 2)
    x ← ARRAY.arange((−radius), (radius + 1), dtype = ARRAY.float64)
    g ← ARRAY.exp(((−(x ^ 2)) / (2.0 * (sigma ^ 2))))
    RETURN (g / SUM(g))
END PROCEDURE
g1 ← build_gaussian_kernel_1d(5, 1.0)
outer ← ARRAY.outer(g1, g1)
CALL EMIT("Maximum difference between 2-D kernel and outer product:")
CALL EMIT(MAX(ARRAY, ARRAY.abs((g5 − outer))))
REQUIRE ARRAY.allclose(g5, outer) OTHERWISE STOP WITH ValidationError
gaussian_filtered_image ← NDIMAGE.gaussian_filter(CAST(einstein_gaussian_noise_image, ARRAY.float32),
    sigma = 1.5, mode = "reflect")
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
CALL axes[0].imshow(einstein_reference_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Reference")
CALL axes[1].imshow(einstein_gaussian_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[1].set_title("Gaussian-noisy")
CALL axes[2].imshow(gaussian_filtered_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Gaussian filter, sigma=1.5")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "04_gaussian_filter.png"), bbox_inches = "tight")
CALL PLOT.show()
sigmas ← [0.5, 1.0, 2.0, 3.0]
gaussian_filter_results ← MAP{sigma ↦ NDIMAGE.gaussian_filter(CAST(einstein_gaussian_noise_image,
    ARRAY.float32), sigma = sigma, mode = "reflect") FOR sigma IN sigmas}
(fig, axes) ← PLOT.subplots(1, 5, figsize = (17, 4))
CALL axes[0].imshow(einstein_gaussian_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Noisy input")
FOR (ax, sigma) IN ZIP(axes[1:], sigmas) DO
    CALL ax.imshow(gaussian_filter_results[sigma], cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(FORMAT("sigma={0}" , sigma))
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "05_gaussian_sigma_sweep.png"), bbox_inches = "tight")
CALL PLOT.show()
mean_5 ← NDIMAGE.uniform_filter(CAST(einstein_gaussian_noise_image, ARRAY.float32), size = 5,
    mode = "reflect")
gauss_approx ← NDIMAGE.gaussian_filter(CAST(einstein_gaussian_noise_image, ARRAY.float32), sigma = 1.0,
    mode = "reflect")
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
FOR (ax, img, title) IN ZIP(axes, [einstein_gaussian_noise_image, mean_5, gauss_approx], ["Noisy", "5x5 mean",
    "Gaussian sigma=1"]) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "06_mean_vs_gaussian.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 12 — 9. Noise-Model Dependence
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
FOR (ax, img, title) IN ZIP(axes, [einstein_reference_image, einstein_gaussian_noise_image,
    einstein_salt_pepper_noise_image, einstein_speckle_noise_image], ["Reference", "Gaussian noise", "Salt-and-pepper",
    "Speckle noise"]) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "07_noise_types.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 13 — 10. Median Filtering
values ← ARRAY.array([20, 21, 20, 22, 255, 19, 20, 21, 20])
CALL EMIT("Values:", values)
CALL EMIT("Sorted:", ARRAY.sort(values))
CALL EMIT("Mean  :", MEAN(values))
CALL EMIT("Median:", ARRAY.median(values))
median_filtered ← NDIMAGE.median_filter(einstein_salt_pepper_noise_image, size = 3, mode = "reflect")
mean_on_salt ← NDIMAGE.uniform_filter(CAST(einstein_salt_pepper_noise_image, ARRAY.float32), size = 3,
    mode = "reflect")
gaussian_on_salt ← NDIMAGE.gaussian_filter(CAST(einstein_salt_pepper_noise_image, ARRAY.float32),
    sigma = 1.0, mode = "reflect")
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
FOR (ax, img, title) IN ZIP(axes, [einstein_salt_pepper_noise_image, mean_on_salt, gaussian_on_salt,
    median_filtered], ["Salt-and-pepper", "Mean", "Gaussian", "Median"]) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "08_median_filter.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 14 — Median-Window Sweep
median_sizes ← [3, 5, 7]
median_results ← MAP{size ↦ NDIMAGE.median_filter(einstein_salt_pepper_noise_image, size = size,
    mode = "reflect") FOR size IN median_sizes}
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
CALL axes[0].imshow(einstein_salt_pepper_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Noisy")
FOR (ax, size) IN ZIP(axes[1:], median_sizes) DO
    CALL ax.imshow(median_results[size], cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(FORMAT("{0}x{1} median" , size, size))
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "09_median_sizes.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 15 — 11. Bilateral Filtering
bilateral ← CV.bilateralFilter(einstein_gaussian_noise_image, d = 9, sigmaColor = 45, sigmaSpace = 45)
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
FOR (ax, img, title) IN ZIP(axes, [einstein_reference_image, einstein_gaussian_noise_image,
    gaussian_filtered_image, bilateral], ["Reference", "Noisy", "Gaussian filter", "Bilateral filter"]) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "10_bilateral_filter.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 16 — Bilateral-Parameter Sweep
sigma_colors ← [10, 30, 60, 120]
bilateral_results ← MAP{sc ↦ CV.bilateralFilter(einstein_gaussian_noise_image, d = 9, sigmaColor = sc,
    sigmaSpace = 45) FOR sc IN sigma_colors}
(fig, axes) ← PLOT.subplots(1, 5, figsize = (17, 4))
CALL axes[0].imshow(einstein_gaussian_noise_image, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Noisy")
FOR (ax, sc) IN ZIP(axes[1:], sigma_colors) DO
    CALL ax.imshow(bilateral_results[sc], cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(FORMAT("sigmaColor={0}" , sc))
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "11_bilateral_parameter_sweep.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 17 — 12. Quantitative Denoising Metrics
PROCEDURE mean_absolute_error(reference, test)
    ref ← CAST(reference, ARRAY.float64)
    tst ← ARRAY.asarray(test, dtype = ARRAY.float64)
    RETURN REAL(MEAN(ARRAY, ARRAY.abs((ref − tst))))
END PROCEDURE
PROCEDURE mean_squared_error(reference, test)
    ref ← CAST(reference, ARRAY.float64)
    tst ← ARRAY.asarray(test, dtype = ARRAY.float64)
    RETURN REAL(MEAN(ARRAY, ((ref − tst) ^ 2)))
END PROCEDURE
PROCEDURE root_mean_squared_error(reference, test)
    RETURN MATH.sqrt(mean_squared_error(reference, test))
END PROCEDURE
PROCEDURE peak_signal_to_noise_ratio(reference, test, peak = 255.0)
    value ← mean_squared_error(reference, test)
    IF (value = 0) THEN
        RETURN REAL("inf")
    END IF
    RETURN (10.0 * MATH.log10(((peak ^ 2) / value)))
END PROCEDURE
PROCEDURE build_metric_summary(name, reference, test)
    RETURN (name, mean_absolute_error(reference, test), mean_squared_error(reference, test),
        root_mean_squared_error(reference, test), peak_signal_to_noise_ratio(reference, test))
END PROCEDURE

# Algorithm 18 — Gaussian-Noise Comparison
gaussian_candidates ← [build_metric_summary("Noisy", einstein_reference_image,
    einstein_gaussian_noise_image), build_metric_summary("Mean 3x3", einstein_reference_image,
    mean_filtered_image), build_metric_summary("Gaussian sigma=1.5", einstein_reference_image,
    gaussian_filtered_image), build_metric_summary("Bilateral", einstein_reference_image, bilateral)]
CALL EMIT(FORMAT("{0:24s} {1:>10s} {2:>12s} {3:>10s} {4:>10s}" , "Method", "MAE", "MSE", "RMSE",
    "PSNR(dB)"))
CALL EMIT(("-" * 72))
FOR (name, a, m, r, p) IN gaussian_candidates DO
    CALL EMIT(FORMAT("{0:24s} {1:10.3f} {2:12.3f} {3:10.3f} {4:10.3f}" , name, a, m, r, p))
END FOR

# Algorithm 19 — Impulse-Noise Comparison
salt_candidates ← [build_metric_summary("Noisy", einstein_reference_image, einstein_salt_pepper_noise_image),
    build_metric_summary("Mean 3x3", einstein_reference_image, mean_on_salt), build_metric_summary("Gaussian sigma=1",
    einstein_reference_image, gaussian_on_salt), build_metric_summary("Median 3x3",
    einstein_reference_image, median_filtered)]
CALL EMIT(FORMAT("{0:24s} {1:>10s} {2:>12s} {3:>10s} {4:>10s}" , "Method", "MAE", "MSE", "RMSE",
    "PSNR(dB)"))
CALL EMIT(("-" * 72))
FOR (name, a, m, r, p) IN salt_candidates DO
    CALL EMIT(FORMAT("{0:24s} {1:10.3f} {2:12.3f} {3:10.3f} {4:10.3f}" , name, a, m, r, p))
END FOR

# Algorithm 20 — 13. Edge Preservation as a Secondary Check
PROCEDURE compute_sobel_gradient_magnitude(image)
    arr ← ARRAY.asarray(image, dtype = ARRAY.float32)
    gx ← NDIMAGE.sobel(arr, axis = 1, mode = "reflect")
    gy ← NDIMAGE.sobel(arr, axis = 0, mode = "reflect")
    RETURN ARRAY.hypot(gx, gy)
END PROCEDURE
edge_ref ← compute_sobel_gradient_magnitude(einstein_reference_image)
edge_mean ← compute_sobel_gradient_magnitude(mean_filtered_image)
edge_gauss ← compute_sobel_gradient_magnitude(gaussian_filtered_image)
edge_bilat ← compute_sobel_gradient_magnitude(bilateral)
CALL EMIT("Mean gradient magnitude")
CALL EMIT(FORMAT("Reference : {0:.3f}" , MEAN(edge_ref)))
CALL EMIT(FORMAT("Mean      : {0:.3f}" , MEAN(edge_mean)))
CALL EMIT(FORMAT("Gaussian  : {0:.3f}" , MEAN(edge_gauss)))
CALL EMIT(FORMAT("Bilateral : {0:.3f}" , MEAN(edge_bilat)))

# Algorithm 21 — 14. Sharpening with the Laplacian
laplacian_kernel ← ARRAY.array([[0, 1, 0], [1, (−4), 1], [0, 1, 0]], dtype = ARRAY.float32)
CALL EMIT(laplacian_kernel)
CALL EMIT("Kernel sum:", SUM(laplacian_kernel))
moon_float ← CAST(moon, ARRAY.float32)
laplacian ← NDIMAGE.convolve(moon_float, laplacian_kernel, mode = "reflect")
lap_sharp ← CAST(ARRAY.clip((moon_float − laplacian), 0, 255), ARRAY.uint8)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
CALL axes[0].imshow(moon, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Blurred moon")
CALL axes[1].imshow(laplacian, cmap = "gray")
CALL axes[1].set_title("Laplacian response")
CALL axes[2].imshow(lap_sharp, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[2].set_title("Laplacian sharpened")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "12_laplacian_sharpening.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 22 — 15. Unsharp Masking and High-Boost Filtering
blurred ← NDIMAGE.gaussian_filter(moon_float, sigma = 2.0, mode = "reflect")
detail_mask ← (moon_float − blurred)
unsharp ← CAST(ARRAY.clip((moon_float + (1.0 * detail_mask)), 0, 255), ARRAY.uint8)
high_boost ← CAST(ARRAY.clip((moon_float + (1.8 * detail_mask)), 0, 255), ARRAY.uint8)
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
FOR (ax, img, title) IN ZIP(axes, [moon, blurred, unsharp, high_boost], ["Input", "Gaussian blur", "Unsharp k=1",
    "High-boost k=1.8"]) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "13_unsharp_highboost.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 23 — 16. First Derivatives and Image Gradients
gx ← NDIMAGE.sobel(CAST(ascent, ARRAY.float32), axis = 1, mode = "reflect")
gy ← NDIMAGE.sobel(CAST(ascent, ARRAY.float32), axis = 0, mode = "reflect")
magnitude ← ARRAY.hypot(gx, gy)
orientation ← ARRAY.arctan2(gy, gx)
(fig, axes) ← PLOT.subplots(1, 4, figsize = (15, 4))
CALL axes[0].imshow(ascent, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Input")
CALL axes[1].imshow(ARRAY.abs(gx), cmap = "gray")
CALL axes[1].set_title("|Gx|")
CALL axes[2].imshow(ARRAY.abs(gy), cmap = "gray")
CALL axes[2].set_title("|Gy|")
CALL axes[3].imshow(magnitude, cmap = "gray")
CALL axes[3].set_title("Gradient magnitude")
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "14_sobel_gradients.png"), bbox_inches = "tight")
CALL PLOT.show()
(fig, ax) ← PLOT.subplots(figsize = (6, 5))
im ← ax.imshow(orientation, cmap = "twilight", vmin = (−ARRAY.pi), vmax = ARRAY.pi)
CALL ax.set_title("Gradient orientation (radians)")
CALL ax.axis("off")
CALL fig.colorbar(im, ax = ax, fraction = 0.046, pad = 0.04)
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "15_gradient_orientation.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 24 — 17. Sobel vs Prewitt vs Scharr
ascent_float_image ← CAST(ascent, ARRAY.float32)
prewitt_x ← NDIMAGE.prewitt(ascent_float_image, axis = 1, mode = "reflect")
prewitt_y ← NDIMAGE.prewitt(ascent_float_image, axis = 0, mode = "reflect")
prewitt_mag ← ARRAY.hypot(prewitt_x, prewitt_y)
sobel_x ← NDIMAGE.sobel(ascent_float_image, axis = 1, mode = "reflect")
sobel_y ← NDIMAGE.sobel(ascent_float_image, axis = 0, mode = "reflect")
sobel_mag ← ARRAY.hypot(sobel_x, sobel_y)
scharr_x ← CV.Scharr(ascent_float_image, CV.CV_32F, 1, 0)
scharr_y ← CV.Scharr(ascent_float_image, CV.CV_32F, 0, 1)
scharr_mag ← ARRAY.hypot(scharr_x, scharr_y)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (12, 4))
FOR (ax, img, title) IN ZIP(axes, [prewitt_mag, sobel_mag, scharr_mag], ["Prewitt", "Sobel", "Scharr"]) DO
    CALL ax.imshow(img, cmap = "gray")
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "16_derivative_operators.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 25 — 18. Border Effects on a Real Image
kernel_15 ← (ARRAY.ones((15, 15), dtype = ARRAY.float32) / (15 * 15))
ascent_float_image ← CAST(ascent, ARRAY.float32)
padding_results ← {"constant" ↦ NDIMAGE.convolve(ascent_float_image, kernel_15, mode = "constant",
    cval = 0.0), "nearest" ↦ NDIMAGE.convolve(ascent_float_image, kernel_15, mode = "nearest"),
    "reflect" ↦ NDIMAGE.convolve(ascent_float_image, kernel_15, mode = "reflect"),
    "wrap" ↦ NDIMAGE.convolve(ascent_float_image, kernel_15, mode = "wrap")}
(fig, axes) ← PLOT.subplots(1, 5, figsize = (17, 4))
CALL axes[0].imshow(ascent, cmap = "gray", vmin = 0, vmax = 255)
CALL axes[0].set_title("Input")
FOR (ax, (name, img)) IN ZIP(axes[1:], ITEMS(padding_results)) DO
    CALL ax.imshow(img, cmap = "gray", vmin = 0, vmax = 255)
    CALL ax.set_title(name)
END FOR
FOR ax IN axes DO
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "17_padding_real_image.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 26 — 19. Filtering RGB Images
taj_gaussian ← NDIMAGE.gaussian_filter(CAST(taj_reference_image, ARRAY.float32), sigma = (1.5, 1.5, 0),
    mode = "reflect")
EVALUATE taj_
bilateral ← CV.bilateralFilter(taj_reference_image, d = 9, sigmaColor = 45, sigmaSpace = 45)
(fig, axes) ← PLOT.subplots(1, 3, figsize = (13, 4))
FOR (ax, img, title) IN ZIP(axes, [taj_reference_image, CAST(ARRAY.clip(taj_gaussian, 0, 255), ARRAY.uint8),
    taj_bilateral], ["RGB reference", "Gaussian RGB", "Bilateral RGB"]) DO
    CALL ax.imshow(img)
    CALL ax.set_title(title)
    CALL ax.axis("off")
END FOR
CALL fig.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "18_color_filtering.png"), bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 27 — 20. Numerical Safety
example_uint8 ← ARRAY.array([0, 10, 250, 255], dtype = ARRAY.uint8)
unsafe ← (example_uint8 + ARRAY.uint8(20))
safe ← (CAST(example_uint8, ARRAY.float32) + 20)
CALL EMIT("Original:", example_uint8)
CALL EMIT("Unsafe uint8 + 20:", unsafe)
CALL EMIT("Safe float + 20 :", safe)
CALL EMIT("Clipped result  :", CAST(ARRAY.clip(safe, 0, 255), ARRAY.uint8))

# Algorithm 28 — 21. Filter Selection Criteria
best_gaussian_noise_method ← MIN(gaussian_candidates[1:], key = FUNCTION(row) ↦ row[2])
best_impulse_noise_method ← MIN(salt_candidates[1:], key = FUNCTION(row) ↦ row[2])
edge_retention ← {"Mean 3x3" ↦ REAL((MEAN(edge_mean) / MAX(MEAN(edge_ref), 1e-12))),
    "Gaussian sigma=1.5" ↦ REAL((MEAN(edge_gauss) / MAX(MEAN(edge_ref), 1e-12))),
    "Bilateral" ↦ REAL((MEAN(edge_bilat) / MAX(MEAN(edge_ref), 1e-12)))}
CALL EMIT("Best Gaussian-noise restoration by MSE:")
CALL EMIT(FORMAT("  {0} (MSE={1:.3f}, PSNR={2:.3f} dB)" , best_gaussian_noise_method[0],
    best_gaussian_noise_method[2], best_gaussian_noise_method[4]))
CALL EMIT("\nBest impulse-noise restoration by MSE:")
CALL EMIT(FORMAT("  {0} (MSE={1:.3f}, PSNR={2:.3f} dB)" , best_impulse_noise_method[0],
    best_impulse_noise_method[2], best_impulse_noise_method[4]))
CALL EMIT("\nRelative mean-gradient retention:")
FOR (method, ratio) IN ITEMS(edge_retention) DO
    CALL EMIT(FORMAT("  {0:20s}: {1:.3f}" , method, ratio))
END FOR

# Algorithm 29 — 22. Standard Spatial-Filtering Workflow
PROCEDURE apply_spatial_filter(image, method = "gaussian", kernel_size = 5, sigma = 1.5)
    image ← ARRAY.asarray(image, dtype = ARRAY.float32)
    IF (image.ndim ≠ 2) THEN
        RAISE ValueError("apply_spatial_filter expects a 2-D grayscale image.")
    END IF
    IF ((kernel_size ≤ 0) OR ((kernel_size MOD 2) = 0)) THEN
        RAISE ValueError("kernel_size must be a positive odd integer.")
    END IF
    IF (method = "mean") THEN
        RETURN NDIMAGE.uniform_filter(image, size = kernel_size, mode = "reflect")
    END IF
    IF (method = "gaussian") THEN
        IF (sigma ≤ 0) THEN
            RAISE ValueError("sigma must be strictly positive.")
        END IF
        RETURN NDIMAGE.gaussian_filter(image, sigma = sigma, mode = "reflect")
    END IF
    IF (method = "median") THEN
        RETURN NDIMAGE.median_filter(image, size = kernel_size, mode = "reflect")
    END IF
    RAISE ValueError("method must be one of: 'mean', 'gaussian', 'median'.")
END PROCEDURE
workflow_result ← apply_spatial_filter(einstein_gaussian_noise_image, method = "gaussian", kernel_size = 5,
    sigma = 1.5)
REQUIRE (workflow_result.shape = einstein_gaussian_noise_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.all(ARRAY.isfinite(workflow_result)) OTHERWISE STOP WITH ValidationError
CALL EMIT("Workflow example:", "Gaussian sigma=1.5", FORMAT("shape={0}" , workflow_result.shape),
    FORMAT("MSE={0:.3f}" , mean_squared_error(einstein_reference_image, workflow_result)))

# Algorithm 30 — 23. Validation Checks
REQUIRE (mean_filtered_image.shape = einstein_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (gaussian_filtered_image.shape = einstein_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (median_filtered.shape = einstein_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (bilateral.shape = einstein_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(SUM(mean_kernel), 1.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(SUM(g5), 1.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(SUM(sobel_x_kernel), 0.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.isclose(SUM(laplacian_kernel), 0.0) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.allclose(manual_convolution_result, scipy_convolution_result) OTHERWISE STOP WITH ValidationError
REQUIRE ARRAY.all((magnitude ≥ 0)) OTHERWISE STOP WITH ValidationError
REQUIRE (taj_gaussian.shape = taj_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE (taj_bilateral.shape = taj_reference_image.shape) OTHERWISE STOP WITH ValidationError
REQUIRE ((MIN(lap_sharp) ≥ 0) AND (MAX(lap_sharp) ≤ 255)) OTHERWISE STOP WITH ValidationError
REQUIRE ((MIN(unsharp) ≥ 0) AND (MAX(unsharp) ≤ 255)) OTHERWISE STOP WITH ValidationError
CALL EMIT("All spatial-filtering validation checks passed.")
REQUIRED_OUTPUTS ← ["01_border_modes.png", "02_mean_filter.png", "03_mean_kernel_sizes.png", "04_gaussian_filter.png",
    "05_gaussian_sigma_sweep.png", "06_mean_vs_gaussian.png", "07_noise_types.png", "08_median_filter.png",
    "09_median_sizes.png", "10_bilateral_filter.png", "11_bilateral_parameter_sweep.png", "12_laplacian_sharpening.png",
    "13_unsharp_highboost.png", "14_sobel_gradients.png", "15_gradient_orientation.png", "16_derivative_operators.png",
    "17_padding_real_image.png", "18_color_filtering.png"]
missing_outputs ← LIST[output_name FOR output_name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / output_name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT(FORMAT("Output validation passed: {0} files." , LENGTH(REQUIRED_OUTPUTS)))

END ALGORITHM
```

## Part II — Step-by-step auditable specification

Each step below is a six-section contract corresponding to the numbered block in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts and shapes required before execution.
- **Procedure** — ordered computation.
- **Produces** — outputs and invariants.
- **Acceptance contract** — numerical or structural gates before the next step.
- **Failure mode** — explicit abort, skip, or diagnostic action; never a silent substitution.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project environment and use this notebook as the coding sequence for `main.ipynb`.
2. Keep boundary handling, source images, and comparison parameters explicit.
3. Use clean references whenever quantitative denoising metrics are computed.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create Paths and Discover Input Images

**Purpose**

Associate each experiment with a known input.

**Inputs**

Lab data files and semantic role patterns.

**Procedure**

1. Import `Path`, math, NumPy, Matplotlib, PIL, SciPy `ndimage`, and OpenCV.
2. Locate the lab root and create data/output paths.
3. Discover supported files recursively.
4. Map semantic roles such as clean reference, Gaussian-noisy, salt-and-pepper, speckle, blurred moon, and RGB examples to deterministic files.

**Produces**

INPUT_IMAGES and output directory.

**Acceptance contract**

Every clean/noisy role resolves deterministically.

**Failure mode**

Stop on missing data; do not silently pair unrelated images.

## 2. Load and Validate Images

**Purpose**

Establish image and channel conventions.

**Inputs**

Resolved paths.

**Procedure**

1. Implement grayscale and RGB image loaders.
2. Load each semantic-role image exactly once.
3. Print shape, dtype, and intensity range for each image.
4. Stop if any required role cannot be resolved.

**Produces**

Grayscale/RGB uint8 arrays.

**Acceptance contract**

Nonempty arrays; corresponding clean/noisy dimensions agree.

**Failure mode**

Correct decoding or role matching before comparisons.

## 3. Create Toy Arrays and Reference Kernels

**Purpose**

Make local filtering arithmetic inspectable.

**Inputs**

Small known image and kernels.

**Procedure**

1. Create small numeric arrays for hand-verifiable filtering examples.
2. Define a normalized mean kernel and an asymmetric kernel.
3. Define Sobel and Laplacian kernels explicitly.
4. Use float arithmetic for kernel computations.

**Produces**

Toy neighborhoods and reference kernels.

**Acceptance contract**

Odd support; smoothing kernels sum to one where intended.

**Failure mode**

Do not normalize derivative kernels as if they were averages.

## 4. Differentiate Correlation from Convolution

**Purpose**

Expose the kernel flip that distinguishes convolution.

**Inputs**

Asymmetric kernel and toy image.

**Procedure**

1. Select one local patch from the toy array.
2. Compute correlation using the asymmetric kernel unchanged.
3. Compute convolution after flipping the kernel in both axes.
4. Compare manual center values with SciPy `correlate` and `convolve`.

**Produces**

Manual correlation/convolution values and SciPy references.

**Acceptance contract**

Compare the same centre pixel; flip both kernel axes only for convolution.

**Failure mode**

Fix kernel orientation if manual and reference values differ.

## 5. Implement Direct 2-D Convolution

**Purpose**

Implement the neighborhood operation explicitly.

**Inputs**

2-D image and odd-size kernel.

**Procedure**

1. Create a helper accepting image and odd-sized kernel.
2. Compute half-kernel padding sizes.
3. Pad the image symmetrically.
4. Flip the kernel to implement convolution.
5. Loop over rows/columns, extract each neighborhood, multiply, sum, and store the result.

**Produces**

Same-shaped floating convolution result.

**Acceptance contract**

Symmetric padding; kernel flipped in two axes; no uint8 accumulation.

**Failure mode**

Reject even kernel sizes under the current helper's convention.

## 6. Validate Manual Convolution Against SciPy

**Purpose**

Verify the manual implementation against a library.

**Inputs**

Manual result and ndimage.convolve with equivalent boundaries.

**Procedure**

1. Convert toy image and kernel to float64.
2. Run the manual implementation and SciPy convolution with equivalent boundary behaviour.
3. Compute maximum absolute difference.
4. Assert agreement to tight floating-point tolerance.

**Produces**

Numerical equality check.

**Acceptance contract**

rtol=1e-10 and atol=1e-12 for this floating toy comparison.

**Failure mode**

Inspect padding and flip conventions before changing tolerance.

## 7. Demonstrate Border Modes

**Purpose**

Show how missing neighbors are synthesized.

**Inputs**

One image/kernel and several border modes.

**Procedure**

1. Create a controlled image with content near the boundary.
2. Apply the same filter under different padding conventions.
3. Plot with identical display limits.
4. Use the comparison to reveal edge artefacts.

**Produces**

Border-mode comparison.

**Acceptance contract**

Kernel unchanged; only boundary convention varies.

**Failure mode**

Document the mode instead of treating border differences as algorithm errors.

## 8. Apply Mean Filtering

**Purpose**

Establish a uniform smoothing baseline.

**Inputs**

Image and averaging kernel.

**Procedure**

1. Create a normalized 3×3 box kernel.
2. Convolve the Gaussian-noisy reference image using reflective boundaries.
3. Compare clean, noisy, and filtered images.
4. Save the baseline mean-filter result.

**Produces**

Mean-filtered image.

**Acceptance contract**

Kernel sums to one; source shape preserved.

**Failure mode**

Investigate brightness shifts from an unnormalized kernel.

## 9. Study Mean-Kernel Size

**Purpose**

Measure the smoothing/detail tradeoff of support size.

**Inputs**

Same source and several odd window sizes.

**Procedure**

1. Repeat mean filtering for sizes 3, 5, and 9.
2. Change only kernel size.
3. Store outputs in a dictionary keyed by size.
4. Save the controlled comparison.

**Produces**

Mean-filter sweep.

**Acceptance contract**

Same border mode and source; label each size.

**Failure mode**

Do not change noise realization during a support-size comparison.

## 10. Build a 2-D Gaussian Kernel

**Purpose**

Create a normalized finite Gaussian kernel.

**Inputs**

Odd size and positive sigma.

**Procedure**

1. Require positive sigma and odd kernel size.
2. Create centered x/y coordinate grids.
3. Evaluate the Gaussian response.
4. Normalize to unit sum and verify the sum is approximately one.

**Produces**

Floating kernel shaped (size,size).

**Acceptance contract**

Nonnegative, symmetric weights with sum near one.

**Failure mode**

Existing assertions reject even size/nonpositive sigma.

## 11. Verify Gaussian Separability

**Purpose**

Verify a Gaussian factorization on the same support.

**Inputs**

1-D and 2-D Gaussian kernels.

**Procedure**

1. Build the equivalent normalized 1-D Gaussian vector.
2. Compute its outer product.
3. Compare that product with the explicit 2-D Gaussian kernel.
4. Assert numerical equivalence.

**Produces**

Outer-product comparison.

**Acceptance contract**

np.allclose confirms separability for matching size/sigma.

**Failure mode**

Align truncation and normalization before comparing kernels.

## 12. Apply Gaussian Filtering and Sigma Sweep

**Purpose**

Relate sigma to blur strength.

**Inputs**

Same image and chosen sigma values.

**Procedure**

1. Filter the Gaussian-noisy image with a representative sigma.
2. Evaluate several sigma values while keeping image and boundary handling fixed.
3. Store outputs by sigma.
4. Save the comparison to expose smoothing-versus-detail trade-off.

**Produces**

Gaussian outputs and sweep figure.

**Acceptance contract**

Fixed source/boundary settings; record sigma per output.

**Failure mode**

Check settings rather than attributing all changes to window size.

## 13. Compare Mean and Gaussian Smoothing

**Purpose**

Compare different weighting within a neighborhood.

**Inputs**

Mean and Gaussian outputs on the same image.

**Procedure**

1. Apply comparable mean and Gaussian smoothing to the same noisy input.
2. Plot noisy input and both filtered results together.
3. Keep display scaling identical.
4. Defer method selection until quantitative metrics are available.

**Produces**

Paired smoothing diagnostic.

**Acceptance contract**

Common source, display range and documented support.

**Failure mode**

Do not claim equal blur solely because support sizes match.

## 14. Inspect the Noise Models

**Purpose**

Identify the corruption model before choosing a filter.

**Inputs**

Clean, Gaussian, impulse and speckle examples.

**Procedure**

1. Display clean reference, Gaussian noise, salt-and-pepper noise, and speckle noise side by side.
2. Use these known noise types to motivate filter choice.
3. Do not compare filters across different noise inputs as if the conditions were identical.

**Produces**

Noise-model overview.

**Acceptance contract**

Correct clean/noisy pairing and matching dimensions.

**Failure mode**

Resolve mismatched files before computing reference metrics.

## 15. Apply Median Filtering to Impulse Noise

**Purpose**

Test a rank-based filter on impulse corruption.

**Inputs**

Salt-and-pepper image and odd median window.

**Procedure**

1. Demonstrate mean versus median on a small sample containing an outlier.
2. Apply a 3×3 median filter to the salt-and-pepper image.
3. Compare mean, Gaussian, and median outputs on the same impulse-noise input.
4. Save the comparison.

**Produces**

Median-filtered image.

**Acceptance contract**

Source shape preserved; display removed impulses and lost detail.

**Failure mode**

Do not interpret median filtering as linear convolution.

## 16. Study Median Window Size

**Purpose**

Observe the detail loss from larger median windows.

**Inputs**

Same noisy image and several odd windows.

**Procedure**

1. Evaluate median sizes 3, 5, and 7 on the same salt-and-pepper image.
2. Store outputs keyed by size.
3. Compare impulse removal against detail loss.
4. Save the size sweep.

**Produces**

Median sweep.

**Acceptance contract**

Only window size varies.

**Failure mode**

Inspect thin-structure removal before choosing the largest window.

## 17. Apply Bilateral Filtering

**Purpose**

Use spatial and intensity distance jointly.

**Inputs**

Image; bilateral diameter, sigmaColor and sigmaSpace.

**Procedure**

1. Apply OpenCV bilateral filtering to the Gaussian-noisy image.
2. Set neighborhood diameter, `sigmaColor`, and `sigmaSpace` explicitly.
3. Compare reference, noisy input, Gaussian smoothing, and bilateral output.
4. Keep the source image fixed.

**Produces**

Bilateral-filtered output.

**Acceptance contract**

Parameters expressed in the input intensity/spatial units.

**Failure mode**

Correct dtype or intensity scale before tuning sigmaColor.

## 18. Study Bilateral Parameters

**Purpose**

Distinguish bilateral parameter effects.

**Inputs**

Fixed source and declared parameter combinations.

**Procedure**

1. Vary `sigmaColor` across the declared values.
2. Keep diameter and `sigmaSpace` fixed.
3. Store and plot each result.
4. Use the sweep to isolate photometric-similarity effects.

**Produces**

Bilateral sweep.

**Acceptance contract**

Record each combination; vary parameters in controlled comparisons.

**Failure mode**

Do not attribute a multi-parameter change to one parameter alone.

## 19. Implement Denoising Metrics

**Purpose**

Quantify denoising against matched clean data.

**Inputs**

Reference and same-shaped estimate.

**Procedure**

1. Create MAE, MSE, RMSE, and PSNR helpers in float arithmetic.
2. Return infinite PSNR only when MSE is exactly zero.
3. Create one metric-summary helper so every method is scored identically.

**Produces**

MAE, MSE and PSNR.

**Acceptance contract**

Float differences; zero MSE yields infinite PSNR.

**Failure mode**

Reject misaligned references; report expected infinite self-PSNR correctly.

## 20. Compare Filters on Gaussian Noise

**Purpose**

Compare filters on additive Gaussian corruption.

**Inputs**

Clean reference and one noisy image.

**Procedure**

1. Score noisy input, mean, Gaussian, and bilateral outputs against the clean reference.
2. Print one aligned metric table.
3. Use the same reference and peak value for every method.
4. Retain results for later method selection.

**Produces**

Per-filter outputs and metric table.

**Acceptance contract**

All scores use the same reference and noise realization.

**Failure mode**

Inspect pairing/ranges if scores contradict displayed inputs.

## 21. Compare Filters on Salt-and-Pepper Noise

**Purpose**

Compare filters on sparse impulse corruption.

**Inputs**

Clean reference and salt-and-pepper image.

**Procedure**

1. Score noisy input, mean, Gaussian, and median outputs against the same clean reference.
2. Print the same metric set and formatting.
3. Keep the noise model fixed while changing only the filter.

**Produces**

Per-filter outputs and metrics.

**Acceptance contract**

Same input/reference; preserve parameter labels.

**Failure mode**

Do not generalize the ranking to all noise models.

## 22. Measure Edge Preservation

**Purpose**

Add a detail diagnostic to reference error.

**Inputs**

Clean/noisy/filtered images.

**Procedure**

1. Implement Sobel gradient magnitude.
2. Compute gradient maps for the clean reference and denoised candidates.
3. Compare mean gradient magnitude as a secondary detail-retention diagnostic.
4. Use this together with pixel-error metrics.

**Produces**

Mean gradient-magnitude summaries.

**Acceptance contract**

Compare like-for-like ranges; noise can also raise gradients.

**Failure mode**

Do not equate a larger gradient score with better denoising.

## 23. Sharpen with the Laplacian

**Purpose**

Enhance local contrast using second derivatives.

**Inputs**

Image and the declared Laplacian kernel.

**Procedure**

1. Convolve the blurred moon with the explicit Laplacian kernel.
2. Combine original and Laplacian according to the chosen sign convention.
3. Clip only the final sharpened image to `[0,255]`.
4. Save input, Laplacian response, and sharpened output.

**Produces**

Laplacian response and sharpened result.

**Acceptance contract**

Use floating/signed response; sharpening sign follows kernel convention.

**Failure mode**

Correct sign or overflow before clipping the displayed result.

## 24. Implement Unsharp and High-Boost Filtering

**Purpose**

Separate low-frequency content from detail.

**Inputs**

Image, blur and gain.

**Procedure**

1. Gaussian-blur the moon to estimate its low-frequency component.
2. Compute detail mask = original − blurred.
3. Create unsharp output with gain 1 and high-boost output with a larger gain.
4. Clip final outputs and save the comparison.

**Produces**

Detail mask and sharpened images.

**Acceptance contract**

detail=image-blur; image+gain*detail; gain zero preserves source.

**Failure mode**

Keep signed detail; clip only display/storage output.

## 25. Compute First-Derivative Gradients

**Purpose**

Measure directional intensity changes.

**Inputs**

Grayscale image and derivative operators.

**Procedure**

1. Compute horizontal and vertical Sobel derivatives.
2. Compute gradient magnitude and orientation.
3. Visualize components, magnitude, and orientation consistently.
4. Keep derivative arrays in float until display.

**Produces**

Gx, Gy, magnitude and orientation.

**Acceptance contract**

Magnitude=sqrt(Gx**2+Gy**2); orientation=atan2(Gy,Gx).

**Failure mode**

Correct coordinate/sign conventions; do not erase negative derivatives.

## 26. Compare Prewitt, Sobel, and Scharr

**Purpose**

Compare derivative discretizations.

**Inputs**

Same image and Prewitt/Sobel/Scharr operators.

**Procedure**

1. Apply each operator to the same float image.
2. Combine x/y components into gradient magnitude.
3. Display all three magnitudes side by side.
4. Keep source and boundary handling fixed.

**Produces**

Gradient comparison.

**Acceptance contract**

Record operator scaling; raw magnitudes need not be equally normalized.

**Failure mode**

Avoid attributing kernel gain differences to superior edge detection.

## 27. Inspect Border Effects on a Real Image

**Purpose**

Expose border choices on natural content.

**Inputs**

Same real image and filtering parameters.

**Procedure**

1. Apply a large averaging kernel with different border modes to the same real image.
2. Focus the comparison on boundary regions.
3. Save the diagnostic showing padding assumptions.

**Produces**

Padding comparison figure.

**Acceptance contract**

Only border mode differs; inspect boundaries separately from interior.

**Failure mode**

Correct mismatched kernels before interpreting edge artifacts.

## 28. Filter RGB Images Safely

**Purpose**

Filter color without mixing channel indices.

**Inputs**

RGB image and spatial operator.

**Procedure**

1. Apply Gaussian filtering with sigma `(spatial_y, spatial_x, 0)` so channels are not mixed.
2. Apply bilateral filtering to the same RGB image.
3. Verify output shape and channel count are unchanged.
4. Save the color-filtering comparison.

**Produces**

RGB filtered image.

**Acceptance contract**

Filter each channel spatially; preserve (height,width,3).

**Failure mode**

Do not smooth across the channel axis accidentally.

## 29. Demonstrate Numerical Safety

**Purpose**

Demonstrate numerical overflow and safe conversion.

**Inputs**

uint8 image and arithmetic example.

**Procedure**

1. Show how direct uint8 arithmetic can overflow or wrap.
2. Repeat the same operation in float or a wider type.
3. Clip to `[0,255]` only before casting back to uint8.
4. Use this rule for all signed intermediate computations.

**Produces**

Unsafe/safe comparison.

**Acceptance contract**

Float computation precedes clipping and casting.

**Failure mode**

Replace overflowing arithmetic before using results downstream.

## 30. Define Filter-Selection Criteria

**Purpose**

Choose a filter from the corruption and preservation goal.

**Inputs**

Observed noise and preceding diagnostics.

**Procedure**

1. Use metric tables and edge-retention diagnostics to identify suitable filters for Gaussian versus impulse noise.
2. Keep the decision conditional on noise model and detail-preservation requirement.
3. Do not replace measured evidence with a generic ranking.

**Produces**

Reasoned method-selection summary.

**Acceptance contract**

Tie each recommendation to shown experiments, not a universal ranking.

**Failure mode**

State uncertainty if clean reference or noise characterization is unavailable.

## 31. Implement a Reusable Spatial-Filtering Workflow

**Purpose**

State the reusable order of operations for a filtering experiment.

**Inputs**

Input role, selected method, boundary mode and parameters.

**Procedure**

1. Select a source image and document its noise model or filtering objective.
2. Convert to a safe computation dtype and choose the operator, boundary mode and parameters.
3. Apply the operator already demonstrated in the notebook.
4. Inspect the output and compute reference metrics only when a valid reference exists.
5. Record parameters, visual evidence and limits. This section specifies a workflow; it does not introduce a generic callable function.

**Produces**

Documented workflow checklist in the main notebook.

**Acceptance contract**

Loading, floating arithmetic, filtering and evaluation follow the stated order.

**Failure mode**

Do not claim a reusable dispatcher function exists; the current section is a written workflow.

## 32. Run Final Validation

**Purpose**

Check convolution identities and generated diagnostics.

**Inputs**

Kernels, convolution results, metrics and files.

**Procedure**

1. Assert major filtered outputs match source dimensions.
2. Verify smoothing-kernel sums and expected zero-sum derivative/sharpening kernels.
3. Recheck manual convolution against SciPy.
4. Verify RGB outputs, intensity ranges, and all 18 required files.
5. Print the final pass message only after all checks succeed.

**Produces**

Final validation outcome.

**Acceptance contract**

Kernel normalization/separability and manual/reference checks pass; required files exist.

**Failure mode**

Report failed invariant or missing output; figure presence alone does not establish denoising quality.